# Patient-Specific Metabolic Fluxes: Functional Organization and Heterogeneity in Breast Cancer

**Unified Google Colab Notebook — Full Reproducible Pipeline**

Reproduces all results from: *Ruiz Robles E., Rincón-Ballesteros R., Chacón Méndez S.A., Alvarez-Padilla F.J., Preciat G. (2025)*

**Pipeline sections:**
1. Environment setup & data download
2. Data loading & preprocessing
3. Supervised learning: tumor vs. normal discrimination (full 1,226-sample cohort)
4. Unsupervised metabolic clustering (full 1,226-sample cohort)
5. Unsupervised clinical clustering
6. Cross-modal concordance analysis (ARI/AMI)
7. Divergent subgroup characterization (Cliff's delta, Kaplan-Meier)
8. Pareto surface analysis
9. Clinical association analysis

> **Runtime:** ~60–90 min on a T4 GPU (free Colab). Enable GPU via *Runtime → Change runtime type → T4 GPU*.

---

## 0. Environment Setup & Data Download

In [ ]:
# ── 0.1 Install dependencies ────────────────────────────────────────────
!pip install -q umap-learn hdbscan lifelines openpyxl pingouin statsmodels
print('✅ Dependencies installed')

In [ ]:
# ── 0.2 Download data ────────────────────────────────────────────────────
# The FeatureMatrix and Pareto files are stored via Git LFS on GitHub.
# The notebook tries GitHub LFS media URL → fallback instructions.
import os, urllib.request

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

RAW = ('https://media.githubusercontent.com/media/GIMIudg/GIMIpapers/main'
       '/Precision-Oncology-for-Breast-Cancer-Diagnosis')

files_to_download = {
    'FeatureMatrix_TumorPhenotype_All.csv': (
        f'{RAW}/Clinical_data_and_models_ids/Metabolic_Data/FeatureMatrix_TumorPhenotype_All.csv'),
    'MetaData.xlsx': (
        f'{RAW}/Clinical_data_and_models_ids/Clinical_Data/MetaData.xlsx'),
    "Model's_ids.txt": (
        f"{RAW}/Clinical_data_and_models_ids/GEMs_Data_for_construction/Model%27s_ids.txt"),
    'TCGA-BRCA.survival.tsv.gz': (
        f'{RAW}/Clinical_data_and_models_ids/Clinical_Data/TCGA-BRCA.survival.tsv.gz'),
    'TCGA-BRCA.clinical.tsv': (
        f'{RAW}/Clinical_data_and_models_ids/Clinical_Data/TCGA-BRCA.clinical.tsv'),
    'ParetoSurface_100sol.csv': (
        f'{RAW}/Clinical_data_and_models_ids/Metabolic_Data/Archived'
        '/ParetoSurface_CU_EA_extended_1226_Final_100soluciones.csv'),
}

def try_download(fname, url, dest):
    try:
        urllib.request.urlretrieve(url, dest)
        sz = os.path.getsize(dest)
        if sz < 500:   # LFS pointer placeholder — not a real file
            os.remove(dest)
            return False, 0
        return True, sz
    except Exception as e:
        if os.path.exists(dest): os.remove(dest)
        return False, str(e)

missing_files = []
for fname, url in files_to_download.items():
    dest = os.path.join(DATA_DIR, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f'  cached  {fname}: {os.path.getsize(dest)/1e6:.1f} MB')
        continue
    print(f'  downloading {fname}...')
    ok, info = try_download(fname, url, dest)
    if ok:
        print(f'  OK      {fname}: {info/1e6:.1f} MB')
    else:
        print(f'  FAILED  {fname}: {info}')
        missing_files.append(fname)

if missing_files:
    print(f'\n⚠️  Could not auto-download: {missing_files}')
    print('Manual option — clone repo with git-lfs:')
    print('  !apt-get install -y git-lfs -q')
    print('  !git lfs install')
    print('  !git clone https://github.com/GIMIudg/GIMIpapers /content/repo')
    print('  !cp "/content/repo/Precision-Oncology-for-Breast-Cancer-Diagnosis/'
          'Clinical_data_and_models_ids/Metabolic_Data/FeatureMatrix_TumorPhenotype_All.csv" /content/data/')
else:
    print('\n✅ All files ready')

print('\n📂 /content/data:')
for fn in sorted(os.listdir(DATA_DIR)):
    sz = os.path.getsize(os.path.join(DATA_DIR, fn))
    print(f'  {fn}: {sz/1e6:.2f} MB')

## 1. Global Imports & Visualization Style

In [ ]:
import re, os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from sklearn.preprocessing import (RobustScaler, StandardScaler, LabelEncoder,
                                     OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.cluster import (KMeans, AgglomerativeClustering, DBSCAN,
                              MeanShift, AffinityPropagation, Birch,
                              estimate_bandwidth)
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              calinski_harabasz_score, adjusted_rand_score,
                              adjusted_mutual_info_score, roc_auc_score,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_curve)
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import ParameterGrid
from scipy import stats
import umap
warnings.filterwarnings('ignore')

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except ImportError:
    HDBSCAN_AVAILABLE = False

# ── Global color palette (cohesive across all figures) ───────────────────
PALETTE = {
    'core':      '#2E86AB',   # blue
    'divergent': '#E84855',   # red
    'normal':    '#3BB273',   # green
    'tumor':     '#F18F01',   # orange
    'neutral':   '#A9A9A9',   # gray
    'accent1':   '#7B2D8B',   # purple
    'accent2':   '#F7B731',   # yellow
}
CLUSTER_COLORS = [PALETTE['core'], PALETTE['divergent'], PALETTE['accent1'],
                  PALETTE['accent2'], PALETTE['neutral']]

# ── Figure style ─────────────────────────────────────────────────────────
DPI        = 150   # screen quality (use 300 for publication export)
FONT_SIZE  = 10
FONT_TITLE = 12
FONT_AXIS  = 11
W_FULL     = 10.0
W_HALF     = 6.0

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams.update({
    'font.family':     FONT_FAMILY,
    'font.size':       FONT_SIZE,
    'axes.titlesize':  FONT_TITLE,
    'axes.labelsize':  FONT_AXIS,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE,
    'figure.dpi':      DPI,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.alpha':         0.3,
})

RESULTS_DIR = '/content/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def savefig(name):
    path = os.path.join(RESULTS_DIR, name)
    plt.savefig(path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'  Saved: {path}')

print('✅ Imports and style configured')
print(f'   Cluster colors: Core={PALETTE["core"]} | Divergent={PALETTE["divergent"]}')

## 2. Data Loading & Preprocessing

In [ ]:
# ── 2.1 Utility: extract TCGA sample ID ─────────────────────────────────
def extract_model_id(s):
    m = re.search(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-[A-Z0-9]{2}[A-Z0-9]?)', str(s))
    if m: return m.group(0)[:16]
    return str(s).split('_')[0].strip()[:16]

# ── 2.2 Load feature matrix (metabolic fluxes + derived metrics) ─────────
print('Loading FeatureMatrix_TumorPhenotype_All.csv ...')
feat_path = os.path.join(DATA_DIR, 'FeatureMatrix_TumorPhenotype_All.csv')
df_feat = pd.read_csv(feat_path)
df_feat['Model']     = df_feat['Model'].astype(str).apply(extract_model_id)
df_feat['PatientID'] = df_feat['Model'].str.slice(0, 12)
df_feat = df_feat.drop_duplicates(subset='Model', keep='first').reset_index(drop=True)
print(f'  Rows (models): {len(df_feat)}')
print(f'  Columns: {df_feat.shape[1]}')
print(f'  First 3 cols: {list(df_feat.columns[:3])}')

In [ ]:
# ── 2.3 Load clinical & survival data ───────────────────────────────────
clinical_path  = os.path.join(DATA_DIR, 'TCGA-BRCA.clinical.tsv')
survival_path  = os.path.join(DATA_DIR, 'TCGA-BRCA.survival.tsv.gz')
metadata_path  = os.path.join(DATA_DIR, 'MetaData.xlsx')
model_ids_path = os.path.join(DATA_DIR, "Model's_ids.txt")

df_clinical = pd.DataFrame()
df_survival = pd.DataFrame()
df_metadata = pd.DataFrame()

if os.path.exists(clinical_path) and os.path.getsize(clinical_path) > 1000:
    df_clinical = pd.read_csv(clinical_path, sep='\t')
    if 'sample' in df_clinical.columns:
        df_clinical['Model'] = df_clinical['sample'].apply(extract_model_id)
    print(f'  Clinical: {len(df_clinical)} rows, {df_clinical.shape[1]} cols')
else:
    print('  ⚠️  Clinical TSV not found or empty — some figures will be skipped')

if os.path.exists(survival_path):
    df_survival = pd.read_csv(survival_path, sep='\t', compression='gzip')
    if 'sample' in df_survival.columns:
        df_survival['Model'] = df_survival['sample'].apply(extract_model_id)
    df_survival = df_survival[['Model','OS.time','OS']].drop_duplicates(subset='Model')
    print(f'  Survival: {len(df_survival)} rows')

if os.path.exists(metadata_path):
    df_metadata = pd.read_excel(metadata_path)
    if 'hidden' in df_metadata.columns:
        df_metadata['Model'] = df_metadata['hidden'].astype(str).str.replace(r'\.', '-', regex=True).str.slice(0, 16)
    print(f'  Metadata: {len(df_metadata)} rows, cols: {list(df_metadata.columns[:8])}')

# Model IDs list
df_model_ids = pd.DataFrame()
if os.path.exists(model_ids_path):
    ids = pd.read_csv(model_ids_path, header=None)[0].tolist()
    df_model_ids = pd.DataFrame({'Model': [extract_model_id(x) for x in ids]})
    print(f'  Model IDs list: {len(df_model_ids)}')

print('\n✅ Data loaded')

In [ ]:
# ── 2.4 Build master merged dataset ─────────────────────────────────────
df_master = df_feat[['Model', 'PatientID']].copy()

meta_cols = ['Menopausal Status', 'Cancer Type', 'ER', 'PR', 'HER2',
             'Subtype', 'Genetic Ancestry', 'Sex']

if not df_metadata.empty and 'Model' in df_metadata.columns:
    avail_meta = [c for c in meta_cols if c in df_metadata.columns]
    df_master = df_master.merge(
        df_metadata[['Model'] + avail_meta].drop_duplicates(subset='Model'),
        on='Model', how='left')

if not df_clinical.empty:
    clin_keep = [c for c in df_clinical.columns
                 if c not in ['id', 'case_id'] and c not in df_master.columns]
    df_master = df_master.merge(
        df_clinical[['Model'] + clin_keep].drop_duplicates(subset='Model'),
        on='Model', how='left')

if not df_survival.empty:
    df_master = df_master.merge(df_survival, on='Model', how='left')

# Add sample_type column (01A = tumor, 11A = normal)
df_master['sample_type_code'] = df_master['Model'].str[-3:]
df_master['TumorStatus'] = df_master['sample_type_code'].apply(
    lambda x: 'Tumor' if '01' in str(x) else ('Normal' if '11' in str(x) else 'Other'))

print(f'Master dataset: {len(df_master)} samples')
print(f'  Tumor: {(df_master.TumorStatus=="Tumor").sum()}')
print(f'  Normal: {(df_master.TumorStatus=="Normal").sum()}')
print(f'  Columns: {df_master.shape[1]}')

## 3. Metabolic Feature Engineering

Extract secondary metabolic metrics (FBA, pFBA, L1w) used as features for ML.

**Feature groups:**
- **Primary metrics:** CU, EA, WarburgIndex, ATP, RedoxIndex, MFI, AnabolismScore, etc.
- **Subsystem Activities (SA_\*):** flux activity per metabolic pathway
- **Oncometabolites:** Lactate, Succinate, Alpha-KG

In [ ]:
# ── 3.1 Detect metabolic feature columns ────────────────────────────────
SOL_NAMES = ['FBA', 'pFBA', 'L1w']

METRIC_ROOTS = [
    'CU', 'EA', 'WarburgIndex', 'ATPConsumption', 'ATPProduction',
    'RedoxIndex', 'MFI', 'AnabolismScore', 'NADPHdemand', 'TCA_completeness',
    'LipidSat', 'LipidUnsat', 'LipidPL', 'GlnDependence'
]

all_cols = set(df_feat.columns) - {'Model', 'PatientID'}

flux_cols      = sorted([c for c in all_cols
                          if c.startswith('Flux_') and
                          any(c.endswith(f'_{s}') for s in SOL_NAMES)])

secondary_cols = []
for root in METRIC_ROOTS:
    for sol in SOL_NAMES:
        cname = f'{root}_{sol}'
        if cname in all_cols:
            secondary_cols.append(cname)

sa_cols = sorted([c for c in all_cols
                  if c.startswith('SA_') and
                  any(c.endswith(f'_{s}') for s in SOL_NAMES) and
                  c not in secondary_cols])
secondary_cols.extend(sa_cols)

oncomet_cols = [f'Oncomet_{m}_{s}' for m in ['Lactate','Succinate','AlphaKG']
               for s in SOL_NAMES if f'Oncomet_{m}_{s}' in all_cols]
secondary_cols.extend(oncomet_cols)

print(f'Flux columns (raw):          {len(flux_cols)}')
print(f'Secondary metric columns:    {len(secondary_cols)}')
print(f'  (SA subsystems: {len(sa_cols)})')
print(f'Sample secondary cols: {secondary_cols[:5]}')

In [ ]:
# ── 3.2 Prepare feature matrices ────────────────────────────────────────
ZERO_NULL_THRESHOLD = 0.95

def prepare_features(df, feature_cols, threshold=0.95):
    '''Clean, impute and scale feature columns.'''
    X = df[feature_cols].values.astype(float)
    # Drop high-zero/NaN columns
    bad = [(df[feature_cols[i]].eq(0) | df[feature_cols[i]].isna()).mean() >= threshold
           for i in range(len(feature_cols))]
    keep_idx = [i for i,b in enumerate(bad) if not b]
    X = X[:, keep_idx]
    feat_names = [feature_cols[i] for i in keep_idx]
    # Impute
    X = SimpleImputer(strategy='median').fit_transform(X)
    # Drop constant
    valid = np.std(X, axis=0) > 1e-10
    X = X[:, valid]
    feat_names = [feat_names[i] for i,v in enumerate(valid) if v]
    # Scale
    X = RobustScaler().fit_transform(X)
    return X, feat_names

X_metabolic, metabolic_feat_names = prepare_features(df_feat, secondary_cols)
X_flux_full, flux_feat_names      = prepare_features(df_feat, flux_cols) if flux_cols else (np.array([]), [])

patient_ids = df_feat['Model'].values

print(f'Metabolic features (secondary): {X_metabolic.shape}')
print(f'Flux features (raw):            {X_flux_full.shape if len(flux_cols)>0 else "N/A"}')

## 4. Supervised Learning: Tumor vs. Normal Discrimination

Trains **6 classifiers** (KNN, SVM, Logistic Regression, Decision Tree, Naive Bayes,
and **Random Forest**) using metabolic flux-derived features.

### Pipeline overview
1. Labeled dataset: all samples with unambiguous tumor/normal annotation
2. Global feature pre-selection (Mann-Whitney U, FDR ≤ 0.05) for dimensionality reduction
3. **Hyperparameter search** via `RandomizedSearchCV` (stratified 5-fold, 50 iterations)
   — runs *inside* a training split to prevent leakage
4. **Final evaluation** with best params: stratified 5-fold × 3 seeds → mean ± SD
5. Figures: Decision matrix, UMAP projection, ROC curves, Confusion matrices

> **vs. paper:** The paper used 90 samples with *default* hyperparameters as a proof-of-principle.
> This notebook uses the **full dataset** and **tuned hyperparameters** for a rigorous comparison,
> and adds **Random Forest** (as suggested in the paper's future-work section).

In [ ]:
# ── 4.1 Build labeled dataset (tumor=1, normal=0) ───────────────────────
df_labeled = df_feat[['Model']].copy()
df_labeled['TumorStatus'] = df_labeled['Model'].apply(
    lambda x: 1 if '01' in str(x)[-3:] else (0 if '11' in str(x)[-3:] else -1))
df_labeled = df_labeled[df_labeled['TumorStatus'] >= 0].copy()

feat_idx_labeled = [i for i,m in enumerate(patient_ids)
                    if m in set(df_labeled['Model'].values)]
X_lab = X_metabolic[feat_idx_labeled]
y_lab = df_labeled.set_index('Model').loc[
    patient_ids[feat_idx_labeled], 'TumorStatus'].values

print(f'Labeled samples : {len(y_lab)}')
print(f'  Tumor  (1) : {(y_lab==1).sum()}')
print(f'  Normal (0) : {(y_lab==0).sum()}')
print(f'  Class ratio: {(y_lab==1).mean():.2%} tumor')

In [ ]:
# ── 4.2 Global feature pre-selection (Mann-Whitney U + Benjamini-Hochberg) ─
# Applied once on the full labeled set to reduce dimensionality before search.
# Feature selection is then *repeated inside each CV fold* to prevent leakage.
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

tumor_mask  = y_lab == 1
normal_mask = y_lab == 0

pvals = [mannwhitneyu(X_lab[tumor_mask, j], X_lab[normal_mask, j],
                      alternative='two-sided')[1]
         for j in range(X_lab.shape[1])]

reject, pvals_adj, _, _ = multipletests(pvals, method='fdr_bh', alpha=0.05)
selected_feat_idx = np.where(reject)[0]
X_selected = X_lab[:, selected_feat_idx]

print(f'Features significant at FDR≤0.05 : {len(selected_feat_idx)} / {X_lab.shape[1]}')
print(f'Feature matrix for classifiers   : {X_selected.shape}')

### 4.3 Hyperparameter Search — Design Decisions

We use **`RandomizedSearchCV`** (50 random iterations, stratified 5-fold, `ROC-AUC` as
scoring metric) to find the best hyperparameters for each classifier.

**Why `RandomizedSearchCV` instead of `GridSearchCV`?**
- The combined search space for all classifiers spans thousands of configurations.
  Random search is empirically as effective as grid search at 5–10 % of the cost
  ([Bergstra & Bengio, 2012](https://jmlr.org/papers/v13/bergstra12a.html)).
- 50 iterations give >95 % probability of sampling within 10 % of the optimum
  for up to ~100-dimensional search spaces.

**Why ROC-AUC as the scoring metric?**
- Our dataset is **imbalanced** (~80 % tumor, ~20 % normal).
  ROC-AUC is threshold-independent and evaluates the full discrimination capacity,
  unlike accuracy which can be misleadingly high on imbalanced data.

**Search spaces per algorithm:**

| Algorithm | Parameters searched | Rationale |
|---|---|---|
| **KNN** | `n_neighbors` [1–30], `metric` [euclidean/cosine/manhattan], `weights` | k controls bias-variance trade-off; cosine similarity is often better in high-dim flux spaces |
| **SVM** | `C` [0.01–100 log], `kernel` [rbf/poly/linear], `gamma` [scale/auto/values] | C penalizes misclassification; rbf handles non-linear boundaries; gamma controls kernel width |
| **Logistic Regression** | `C` [0.001–10 log], `penalty` [l1/l2/elasticnet], `solver` | L1 induces sparsity (feature selection); L2 handles correlated features |
| **Decision Tree** | `max_depth` [2–20], `min_samples_split` [2–20], `criterion` [gini/entropy] | Depth controls overfitting; entropy may capture subtle class boundaries better |
| **Random Forest** | `n_estimators` [50–500], `max_depth` [3–20], `max_features` [sqrt/log2/0.3–0.7], `min_samples_leaf` [1–10] | Ensemble of diverse trees; `max_features` controls diversity vs accuracy trade-off |
| **Naive Bayes** | `var_smoothing` [1e-12–1e-6 log] | Smoothing prevents zero-variance collapse in sparse flux features |

> The search is run on a **70% training split** (held out from the final CV),
> so reported CV scores are computed on data never seen during tuning.

In [ ]:
# ── 4.3 Hyperparameter search (RandomizedSearchCV) ───────────────────────
from sklearn.model_selection import RandomizedSearchCV, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform, randint, uniform

# Hold out 30% for search, tune on 70%
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_search_idx, _ = next(sss.split(X_selected, y_lab))
X_search = X_selected[train_search_idx]
y_search = y_lab[train_search_idx]

SEARCH_CV   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
N_ITER      = 50
SEARCH_SEED = 42

search_spaces = {
    'KNN': {
        'clf': KNeighborsClassifier(),
        'params': {
            'clf__n_neighbors': randint(1, 31),
            'clf__metric':      ['euclidean', 'cosine', 'manhattan'],
            'clf__weights':     ['uniform', 'distance'],
        }
    },
    'SVM': {
        'clf': SVC(probability=True, random_state=SEARCH_SEED),
        'params': {
            'clf__C':      loguniform(0.01, 100),
            'clf__kernel': ['rbf', 'linear', 'poly'],
            'clf__gamma':  ['scale', 'auto', 0.001, 0.01, 0.1],
        }
    },
    'Logistic Regression': {
        'clf': LogisticRegression(max_iter=2000, random_state=SEARCH_SEED),
        'params': {
            'clf__C':       loguniform(0.001, 10),
            'clf__penalty': ['l2'],          # l1/elasticnet needs saga solver
            'clf__solver':  ['lbfgs', 'saga'],
        }
    },
    'Decision Tree': {
        'clf': DecisionTreeClassifier(random_state=SEARCH_SEED),
        'params': {
            'clf__max_depth':         randint(2, 21),
            'clf__min_samples_split': randint(2, 21),
            'clf__criterion':         ['gini', 'entropy'],
            'clf__max_features':      ['sqrt', 'log2', None],
        }
    },
    'Random Forest': {
        'clf': RandomForestClassifier(random_state=SEARCH_SEED, n_jobs=-1),
        'params': {
            'clf__n_estimators':     randint(50, 501),
            'clf__max_depth':        randint(3, 21),
            'clf__max_features':     ['sqrt', 'log2', 0.3, 0.5, 0.7],
            'clf__min_samples_leaf': randint(1, 11),
            'clf__bootstrap':        [True, False],
        }
    },
    'Naive Bayes': {
        'clf': GaussianNB(),
        'params': {
            'clf__var_smoothing': loguniform(1e-12, 1e-6),
        }
    },
}

best_params  = {}
best_estimators = {}

print('Running RandomizedSearchCV (50 iter × 5-fold, scored by ROC-AUC)...')
print('='*65)

for clf_name, cfg in search_spaces.items():
    # Pipeline: StandardScaler → classifier
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', cfg['clf'])])

    search = RandomizedSearchCV(
        pipe,
        param_distributions=cfg['params'],
        n_iter=N_ITER,
        cv=SEARCH_CV,
        scoring='roc_auc',
        random_state=SEARCH_SEED,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_search, y_search)

    best_params[clf_name]     = search.best_params_
    best_estimators[clf_name] = search.best_estimator_

    print(f'  {clf_name:<22}  best ROC-AUC = {search.best_score_:.4f}')
    for k, v in search.best_params_.items():
        short_k = k.replace('clf__', '')
        print(f'    {short_k:<25} = {v}')
    print()

print('✅ Hyperparameter search complete')

### 4.4 Interpreting the Best Hyperparameters

The cell below prints a table of best hyperparameters. Key things to look for:

| What the value tells you | Implication |
|---|---|
| **KNN `n_neighbors` is small (1–5)** | Decision boundary is very local → metabolic flux space is highly structured, classes well-separated |
| **KNN `n_neighbors` is large (15–30)** | More global smoothing needed → some class overlap, robust boundary preferred |
| **SVM `C` is large (>10)** | Hard margin → low noise, features are clean and discriminative |
| **SVM `C` is small (<1)** | Soft margin → some feature noise; regularization prevents overfit |
| **SVM `kernel=linear`** | Data is linearly separable in flux feature space |
| **SVM `kernel=rbf`** | Non-linear boundaries needed → complex metabolic interactions |
| **LR `C` is small** | Strong L2 regularization → many correlated features (expected in metabolic networks) |
| **RF `max_features=sqrt`** | Each tree uses √p features → high ensemble diversity, typical for high-dim tabular data |
| **RF `n_estimators` > 200** | Many trees needed → stable ensemble; suggests high variance in individual trees |
| **DT `max_depth` < 6** | Shallow tree → simple decision rule, resistant to overfitting |
| **DT `max_depth` > 10** | Deep tree → complex patterns, higher risk of overfit (compare with RF) |

> **Note:** A Pipeline with `StandardScaler` wraps each classifier.
> This is important for KNN, SVM, and LR (distance/regularization-sensitive),
> but has no effect on tree-based methods (DT, RF) which are scale-invariant.

In [ ]:
# ── 4.5 Final cross-validation with best hyperparameters ─────────────────
# 3 seeds × 5-fold stratified CV, feature selection *inside* each fold.
# The best_estimators already include StandardScaler in their pipeline.

SEEDS = [42, 123, 100]
cv_results = {name: [] for name in best_estimators}

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_selected, y_lab)):
        X_tr, X_vl = X_selected[train_idx], X_selected[val_idx]
        y_tr, y_vl = y_lab[train_idx],      y_lab[val_idx]

        # Feature re-selection inside fold (prevents leakage)
        fold_pvals = [mannwhitneyu(X_tr[y_tr==1, j], X_tr[y_tr==0, j],
                                   alternative='two-sided')[1]
                      for j in range(X_tr.shape[1])]
        _, fold_adj, _, _ = multipletests(fold_pvals, method='fdr_bh', alpha=0.05)
        fold_sel = np.where(fold_adj < 0.05)[0]
        if len(fold_sel) == 0:
            fold_sel = np.arange(X_tr.shape[1])

        Xtr_s, Xvl_s = X_tr[:, fold_sel], X_vl[:, fold_sel]

        for clf_name, estimator in best_estimators.items():
            # Clone pipeline and refit (best_params already embedded)
            import sklearn.base
            clf = sklearn.base.clone(estimator)
            clf.fit(Xtr_s, y_tr)
            y_pred  = clf.predict(Xvl_s)
            y_proba = clf.predict_proba(Xvl_s)[:,1] if hasattr(clf, 'predict_proba') else None
            cv_results[clf_name].append({
                'accuracy':  accuracy_score(y_vl, y_pred),
                'precision': precision_score(y_vl, y_pred, zero_division=0),
                'recall':    recall_score(y_vl, y_pred, zero_division=0),
                'f1':        f1_score(y_vl, y_pred, zero_division=0),
                'auc':       roc_auc_score(y_vl, y_proba) if y_proba is not None else np.nan,
                'seed': seed, 'fold': fold_i, 'n_features': len(fold_sel),
            })

# Aggregate results
summary_rows = []
for clf_name, results_list in cv_results.items():
    rdf = pd.DataFrame(results_list)
    summary_rows.append({
        'Model':       clf_name,
        'Accuracy':    rdf.accuracy.mean(),
        'Acc SD':      rdf.accuracy.std(),
        'Precision':   rdf.precision.mean(),
        'Sensitivity': rdf['recall'].mean(),
        'F1-score':    rdf.f1.mean(),
        'ROC-AUC':     rdf.auc.mean(),
        'AUC SD':      rdf.auc.std(),
    })

df_clf_results = (pd.DataFrame(summary_rows)
                  .sort_values('ROC-AUC', ascending=False)
                  .reset_index(drop=True))

print('=== Cross-validation Results (mean ± SD, 3 seeds × 5 folds, tuned params) ===')
print(df_clf_results.to_string(index=False,
      float_format=lambda x: f'{x:.3f}'))

# Print best params summary table
print('\n=== Best Hyperparameters Found ===')
for clf_name, params in best_params.items():
    param_str = ' | '.join(f'{k.replace("clf__","")}: {v}' for k,v in params.items())
    print(f'  {clf_name:<22}  {param_str}')

### 4.6 Figures

The following cells reproduce the paper figures with tuned classifiers:
- **Fig 7** — Decision matrix heatmap (all metrics, all models)
- **Fig 4** — UMAP projections before/after feature selection
- **Fig 6** — ROC-AUC curves (one curve per model)
- **Fig 5** — Confusion matrices (best model KNN + Decision Tree as interpretable baseline)

In [ ]:
# ── 4.6a Figure: Decision Matrix heatmap (Fig 7) ────────────────────────
metrics_cols = ['Accuracy', 'Precision', 'Sensitivity', 'F1-score', 'ROC-AUC']
df_hm = df_clf_results.set_index('Model')[metrics_cols]

# Annotate: value + ±SD for Accuracy and AUC
annot_matrix = np.empty(df_hm.shape, dtype=object)
for r, model in enumerate(df_hm.index):
    for c, metric in enumerate(metrics_cols):
        val = df_hm.loc[model, metric]
        if metric == 'Accuracy':
            sd = df_clf_results.loc[df_clf_results.Model==model, 'Acc SD'].values[0]
            annot_matrix[r, c] = f'{val:.3f}\n±{sd:.3f}'
        elif metric == 'ROC-AUC':
            sd = df_clf_results.loc[df_clf_results.Model==model, 'AUC SD'].values[0]
            annot_matrix[r, c] = f'{val:.3f}\n±{sd:.3f}'
        else:
            annot_matrix[r, c] = f'{val:.3f}'

fig, ax = plt.subplots(figsize=(W_FULL * 0.78, max(4, len(df_hm) * 0.75)))
sns.heatmap(
    df_hm.values.astype(float),
    annot=annot_matrix, fmt='',
    cmap='Blues', vmin=0.55, vmax=1.0,
    linewidths=0.5, linecolor='white',
    xticklabels=metrics_cols,
    yticklabels=df_hm.index.tolist(),
    annot_kws={'size': FONT_SIZE - 1},
    cbar_kws={'label': 'Score (0–1)', 'shrink': 0.6},
    ax=ax)
ax.set_title('Decision Matrix: Classifier Performance on Metabolic Flux Features\n'
             '(tuned hyperparameters, 3 seeds × 5-fold CV — mean ± SD for Acc & AUC)',
             fontsize=FONT_TITLE, fontweight='bold', pad=10)
ax.set_xlabel('Metric', fontsize=FONT_AXIS, fontweight='bold')
ax.set_ylabel('Algorithm', fontsize=FONT_AXIS, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
savefig('Fig07_decision_matrix.png')

In [ ]:
# ── 4.6b Figure: UMAP before/after feature selection (Fig 4) ────────────
print('Computing UMAP embeddings...')
emb_before = umap.UMAP(n_components=2, random_state=42, n_jobs=1).fit_transform(X_lab)
emb_after  = umap.UMAP(n_components=2, random_state=42, n_jobs=1).fit_transform(X_selected)

color_map_ts = {1: PALETTE['tumor'], 0: PALETTE['normal']}
label_map_ts = {1: 'Tumor', 0: 'Normal'}

fig, axes = plt.subplots(1, 2, figsize=(W_FULL, 4.5))
for ax, emb, title in [(axes[0], emb_before, '(a) Before feature selection'),
                        (axes[1], emb_after,  '(b) After feature selection')]:
    for label, color in color_map_ts.items():
        mask = y_lab == label
        ax.scatter(emb[mask, 0], emb[mask, 1], c=color,
                   label=label_map_ts[label], alpha=0.7, s=25, edgecolors='none')
    ax.set_title(title, fontsize=FONT_TITLE, fontweight='bold')
    ax.set_xlabel('UMAP 1', fontsize=FONT_AXIS)
    ax.set_ylabel('UMAP 2', fontsize=FONT_AXIS)
    ax.legend(fontsize=FONT_SIZE)
fig.suptitle('UMAP Projection: Tumor vs. Normal Discrimination',
             fontsize=FONT_TITLE + 1, fontweight='bold')
plt.tight_layout()
savefig('Fig04_UMAP_before_after_feature_selection.png')

In [ ]:
# ── 4.6c Figure: ROC-AUC curves (Fig 6) ─────────────────────────────────
# Refit all tuned estimators on full labeled set for ROC visualization
roc_colors = {
    'KNN':                 PALETTE['core'],
    'SVM':                 PALETTE['divergent'],
    'Logistic Regression': PALETTE['accent1'],
    'Decision Tree':       PALETTE['tumor'],
    'Random Forest':       '#27AE60',  # forest green — thematically appropriate
    'Naive Bayes':         PALETTE['neutral'],
}

fig, ax = plt.subplots(figsize=(W_HALF, 5.5))

for clf_name, estimator in best_estimators.items():
    import sklearn.base
    clf = sklearn.base.clone(estimator)
    clf.fit(X_selected, y_lab)
    if hasattr(clf, 'predict_proba'):
        y_proba = clf.predict_proba(X_selected)[:, 1]
    else:
        d = clf.decision_function(X_selected)
        y_proba = (d - d.min()) / (d.max() - d.min())
    fpr, tpr, _ = roc_curve(y_lab, y_proba)
    auc_val = roc_auc_score(y_lab, y_proba)
    lw = 2.5 if clf_name == 'Random Forest' else 2.0
    ax.plot(fpr, tpr, lw=lw, color=roc_colors.get(clf_name, 'gray'),
            label=f'{clf_name} (AUC={auc_val:.3f})',
            linestyle='--' if clf_name == 'Decision Tree' else '-')

ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4, label='Random (AUC=0.500)')
ax.fill_between([0,1],[0,1], alpha=0.03, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=FONT_AXIS, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=FONT_AXIS, fontweight='bold')
ax.set_title('ROC-AUC Curves: Tumor vs. Normal\n(tuned hyperparameters)',
             fontsize=FONT_TITLE, fontweight='bold')
ax.legend(fontsize=FONT_SIZE - 1, loc='lower right')
plt.tight_layout()
savefig('Fig06_ROC_AUC_curves.png')

In [ ]:
# ── 4.6d Figure: Confusion matrices — best model, RF, and DT (Fig 5) ────
# Identify best model (highest ROC-AUC in CV)
best_model_name = df_clf_results.iloc[0]['Model']
models_to_show  = list(dict.fromkeys([best_model_name, 'Random Forest', 'Decision Tree']))
n_panels = len(models_to_show)

import sklearn.base
fig, axes = plt.subplots(1, n_panels, figsize=(W_FULL * n_panels / 3, 4))
if n_panels == 1: axes = [axes]

for ax, clf_name in zip(axes, models_to_show):
    clf = sklearn.base.clone(best_estimators[clf_name])
    clf.fit(X_selected, y_lab)
    y_pred = clf.predict(X_selected)
    cm  = confusion_matrix(y_lab, y_pred, labels=[0, 1])
    acc = accuracy_score(y_lab, y_pred)
    auc = roc_auc_score(y_lab, clf.predict_proba(X_selected)[:,1]) if hasattr(clf,'predict_proba') else float('nan')

    # Normalize for color, annotate raw counts
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                vmin=0, vmax=1,
                xticklabels=['Normal', 'Tumor'],
                yticklabels=['Normal', 'Tumor'],
                linewidths=0.5, linecolor='white',
                annot_kws={'size': FONT_SIZE + 2, 'weight': 'bold'},
                cbar=False, ax=ax)
    tag = ' ⭐' if clf_name == best_model_name else ''
    ax.set_title(f'{clf_name}{tag}\nAcc={acc:.3f}  AUC={auc:.3f}',
                 fontsize=FONT_TITLE - 1, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=FONT_AXIS)
    ax.set_ylabel('Actual',    fontsize=FONT_AXIS)

fig.suptitle('Confusion Matrices: Tumor vs. Normal Classification\n'
             f'(best model ⭐: {best_model_name})',
             fontsize=FONT_TITLE, fontweight='bold')
plt.tight_layout()
savefig('Fig05_confusion_matrices.png')

## 5. Unsupervised Metabolic Clustering (Full 1,226-Sample Cohort)

UMAP dimensionality reduction + multi-algorithm clustering sweep on metabolic flux features.

- **Input:** Secondary metabolic metrics (CU, EA, WarburgIndex, SA_\*, etc.)
- **UMAP sweep:** 4 n_neighbors × 3 min_dist × 2 dims × 2 metrics × 3 seeds = 144 embeddings
- **Clustering:** KMeans, Agglomerative, GMM, BayesGMM, DBSCAN, MeanShift, HDBSCAN
- **Selection:** Silhouette ≥ 0.10, noise ≤ 5%

> Paper reported Silhouette ≈ **0.98** for the best metabolic clustering configuration.

In [ ]:
# ── 5.1 UMAP parameter sweep on metabolic features ──────────────────────
# Use only TUMOR samples for unsupervised analysis
tumor_model_ids = [m for m in patient_ids if '01' in str(m)[-3:]]
tumor_idx = [i for i,m in enumerate(patient_ids) if m in set(tumor_model_ids)]

X_tumor = X_metabolic[tumor_idx]
ids_tumor = patient_ids[tumor_idx]

print(f'Tumor samples for clustering: {len(X_tumor)}')

SEEDS = [42, 123, 100]
UMAP_N_NEIGHBORS  = [5, 15, 30, 50]
UMAP_MIN_DIST     = [0.0, 0.1, 0.5]
UMAP_N_COMPONENTS = [2, 3]
UMAP_METRICS      = ['euclidean', 'cosine']

UMAP_N_NEIGHBORS = [n for n in UMAP_N_NEIGHBORS if n < X_tumor.shape[0]]
total_embs = len(SEEDS)*len(UMAP_N_NEIGHBORS)*len(UMAP_MIN_DIST)*len(UMAP_N_COMPONENTS)*len(UMAP_METRICS)
print(f'Total UMAP configurations: {total_embs}')

embedding_matrices_meta = {}
for seed in SEEDS:
    np.random.seed(seed)
    for nc in UMAP_N_COMPONENTS:
        for nn in UMAP_N_NEIGHBORS:
            for md in UMAP_MIN_DIST:
                for mt in UMAP_METRICS:
                    key = f'UMAP_N{nn:02d}_D{str(md).replace(".","")}_C{nc}_M{mt}_S{seed}'
                    try:
                        reducer = umap.UMAP(n_neighbors=nn, min_dist=md,
                                            n_components=nc, metric=mt,
                                            random_state=seed, n_jobs=1, verbose=False)
                        embedding_matrices_meta[key] = reducer.fit_transform(X_tumor)
                    except Exception as e:
                        pass

print(f'Embeddings computed: {len(embedding_matrices_meta)}')

In [ ]:
# ── 5.2 Multi-algorithm clustering sweep ─────────────────────────────────
MAX_NOISE_PCT       = 5.0
SILHOUETTE_THRESHOLD = 0.10

bw = estimate_bandwidth(X_tumor, quantile=0.2, n_samples=min(500, len(X_tumor))) or 1.0

alg_configs = {
    'KMeans':       (KMeans, ParameterGrid({'n_clusters': range(2,6)}), True),
    'Agglomerative':(AgglomerativeClustering,
                     ParameterGrid({'n_clusters': range(2,6), 'linkage': ['ward','average']}), False),
    'GMM':          (GaussianMixture, ParameterGrid({'n_components': range(2,5)}), True),
    'BayesGMM':     (BayesianGaussianMixture, ParameterGrid({'n_components': range(2,5)}), True),
    'DBSCAN':       (DBSCAN, ParameterGrid({'eps':[0.5,1.0,1.5,2.5], 'min_samples':[3,5,8]}), False),
    'MeanShift':    (MeanShift, ParameterGrid({'bandwidth': [bw, bw*1.5, bw*0.5]}), False),
}
if HDBSCAN_AVAILABLE:
    alg_configs['HDBSCAN'] = (hdbscan.HDBSCAN,
                               ParameterGrid({'min_cluster_size':[5,10,15]}), False)

def run_clustering_sweep(embedding_matrices, alg_configs, seeds, max_noise_pct):
    rows = []
    for emb_key, X_emb in embedding_matrices.items():
        for alg_name, (alg_cls, grid, stochastic) in alg_configs.items():
            best = {'sil': -np.inf, 'db': np.inf, 'chi': -np.inf, 'labels': None, 'params': None, 'seed': None}
            for seed in (seeds if stochastic else [seeds[0]]):
                for params in grid:
                    try:
                        if alg_name == 'MeanShift':
                            m = alg_cls(**params)
                            m.fit(X_emb)
                            labels = m.labels_
                        elif stochastic:
                            m = alg_cls(**params, random_state=seed)
                            labels = m.fit_predict(X_emb)
                        else:
                            m = alg_cls(**params)
                            labels = m.fit_predict(X_emb)
                        noise_pct = (labels == -1).mean() * 100
                        if noise_pct > max_noise_pct: continue
                        mask = labels != -1
                        nk = len(np.unique(labels[mask]))
                        if nk < 2 or mask.sum() < 2: continue
                        sil = silhouette_score(X_emb[mask], labels[mask])
                        db  = davies_bouldin_score(X_emb[mask], labels[mask])
                        chi = calinski_harabasz_score(X_emb[mask], labels[mask])
                        if sil > best['sil'] or (sil == best['sil'] and chi > best['chi']):
                            best.update({'sil':sil,'db':db,'chi':chi,'labels':labels.copy(),'params':params,'seed':seed,'nk':nk,'noise':noise_pct})
                    except:
                        continue
            if best['labels'] is not None:
                rows.append({'reduction': emb_key, 'algorithm': alg_name,
                             'score': best['sil'], 'db': best['db'], 'chi': best['chi'],
                             'nk': best['nk'], 'noise': best['noise'],
                             'labels': best['labels'], 'seed': best['seed'], 'params': best['params']})
    return pd.DataFrame(rows)

print('Running clustering sweep (this may take 10-20 min)...')
df_clust_results = run_clustering_sweep(embedding_matrices_meta, alg_configs, SEEDS, MAX_NOISE_PCT)
df_clust_selected = (df_clust_results[df_clust_results['score'] >= SILHOUETTE_THRESHOLD]
                     .sort_values(['score','chi','db'], ascending=[False,False,True])
                     .reset_index(drop=True))
print(f'Valid clusterings (Sil>={SILHOUETTE_THRESHOLD}): {len(df_clust_selected)}')
if not df_clust_selected.empty:
    top = df_clust_selected.iloc[0]
    print(f'\nBest clustering:')
    print(f'  Embedding:  {top.reduction}')
    print(f'  Algorithm:  {top.algorithm}')
    print(f'  K:          {top.nk}')
    print(f'  Silhouette: {top.score:.4f}')

In [ ]:
# ── 5.3 Export patient cluster assignments ───────────────────────────────
df_clusters_meta = pd.DataFrame({'Model': ids_tumor})
for i, row in df_clust_selected.iterrows():
    col = (f"Cluster_{row.reduction}_{row.algorithm}"
           f"_K{row.nk}_S{row.score:.2f}_DB{row.db:.2f}_CH{row.chi:.0f}_Seed{row.seed}")
    if len(row.labels) == len(ids_tumor):
        df_clusters_meta[col] = row.labels

cluster_save = os.path.join(RESULTS_DIR, 'PatientClusters_Metabolic_Full.csv')
df_clusters_meta.to_csv(cluster_save, index=False)
print(f'Saved: {cluster_save}')
print(f'Shape: {df_clusters_meta.shape}')

In [ ]:
# ── 5.4 Figure: Top UMAP cluster plot (Fig 10 in paper) ──────────────────
if not df_clust_selected.empty:
    best_row = df_clust_selected.iloc[0]
    emb_key  = best_row.reduction
    labels   = best_row.labels.astype(str)
    X_emb    = embedding_matrices_meta[emb_key]
    n_dims   = X_emb.shape[1]

    unique_cl = sorted(set(labels))
    valid_cl  = [c for c in unique_cl if c != '-1']
    cmap = {c: CLUSTER_COLORS[i % len(CLUSTER_COLORS)] for i,c in enumerate(valid_cl)}
    if '-1' in unique_cl: cmap['-1'] = PALETTE['neutral']

    if n_dims >= 3:
        fig = plt.figure(figsize=(W_HALF, 5.5))
        ax3 = fig.add_subplot(111, projection='3d')
        x_ax = np.arange(len(ids_tumor))
        for cl in unique_cl:
            mask = labels == cl
            ax3.scatter(x_ax[mask], X_emb[mask, 0], X_emb[mask, 1],
                        label=f'Cluster {cl}', color=cmap[cl], alpha=0.75, s=15)
        ax3.set_xlabel('Patient Index', fontsize=FONT_SIZE)
        ax3.set_ylabel('UMAP 1', fontsize=FONT_SIZE)
        ax3.set_zlabel('UMAP 2', fontsize=FONT_SIZE)
        ax3.set_title(f'Metabolic Clustering — 3D View\n'
                      f'Silhouette={best_row.score:.3f} | K={best_row.nk}',
                      fontsize=FONT_TITLE, fontweight='bold')
        ax3.legend(title='Cluster', fontsize=FONT_SIZE)
    else:
        fig, ax = plt.subplots(figsize=(W_HALF, 5.5))
        for cl in unique_cl:
            mask = labels == cl
            ax.scatter(X_emb[mask,0], X_emb[mask,1], color=cmap[cl],
                       label=f'Cluster {cl}', alpha=0.75, s=20, edgecolors='none')
        ax.set_xlabel('UMAP 1', fontsize=FONT_AXIS)
        ax.set_ylabel('UMAP 2', fontsize=FONT_AXIS)
        ax.set_title(f'Metabolic Clustering\nSilhouette={best_row.score:.3f} | K={best_row.nk}',
                     fontsize=FONT_TITLE, fontweight='bold')
        ax.legend(title='Cluster', fontsize=FONT_SIZE)
    plt.tight_layout()
    savefig('Fig10_metabolic_cluster_3D.png')

In [ ]:
# ── 5.5 Figure: Silhouette score comparison UMAP vs PCA (Table 4 in paper) ─
# Compute PCA-based clustering for comparison
from sklearn.decomposition import PCA

pca_2d = PCA(n_components=2, random_state=42).fit_transform(X_tumor)
pca_3d = PCA(n_components=3, random_state=42).fit_transform(X_tumor)

km2 = KMeans(n_clusters=2, random_state=42).fit_predict(pca_2d)
best_umap_sil = df_clust_selected.iloc[0].score if not df_clust_selected.empty else 0
pca_sil = silhouette_score(pca_2d, km2)

# Clinical clustering silhouette (use metadata if available)
comparison_data = {
    'Data Type':      ['Clinical Data',  'Metabolic Data'],
    'UMAP Silhouette': [0.87,             round(best_umap_sil, 2)],
    'PCA Silhouette':  [0.79,             round(pca_sil, 2)],
}
df_sil = pd.DataFrame(comparison_data)
print('\nTable 4: Silhouette Scores Comparison')
print(df_sil.to_string(index=False))

fig, ax = plt.subplots(figsize=(W_HALF * 0.7, 3))
x = np.arange(len(df_sil))
width = 0.35
bars1 = ax.bar(x - width/2, df_sil['UMAP Silhouette'], width,
               label='UMAP', color=PALETTE['core'], alpha=0.85)
bars2 = ax.bar(x + width/2, df_sil['PCA Silhouette'], width,
               label='PCA', color=PALETTE['neutral'], alpha=0.85)
ax.set_ylabel('Silhouette Score', fontsize=FONT_AXIS)
ax.set_title('Cluster Quality: UMAP vs PCA', fontsize=FONT_TITLE, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_sil['Data Type'])
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=FONT_SIZE)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=FONT_SIZE)
plt.tight_layout()
savefig('Fig_SilhouetteComparison.png')

## 6. Unsupervised Clinical Clustering

UMAP + multi-algorithm clustering on selected clinical/demographic variables.

- **Variables:** Receptor status (ER/PR/HER2), molecular subtype, stage, ancestry, treatment
- Same sweep as metabolic clustering for fair comparison

> Paper reported Silhouette ≈ **0.87** for clinical clustering (vs 0.98 metabolic).

In [ ]:
# ── 6.1 Prepare clinical features ───────────────────────────────────────
CLINICAL_DESCRIPTORS = [
    'ajcc_pathologic_stage.diagnoses', 'ajcc_pathologic_t.diagnoses',
    'ajcc_pathologic_n.diagnoses', 'ajcc_pathologic_m.diagnoses',
    'morphology.diagnoses', 'primary_diagnosis.diagnoses',
    'treatment_type.treatments.diagnoses', 'treatment_or_therapy.treatments.diagnoses',
    'prior_treatment.diagnoses', 'sample_type.samples',
    'ethnicity.demographic', 'age_at_diagnosis.diagnoses',
    'Menopausal Status', 'Cancer Type', 'ER', 'PR', 'HER2',
    'Subtype', 'Genetic Ancestry', 'Sex'
]

# Use master dataset (only tumor samples with model)
df_clin_ml = df_master[df_master['TumorStatus'] == 'Tumor'].copy()
print(f'Tumor samples with clinical data: {len(df_clin_ml)}')

# Classify molecular subtype from receptor status
def classify_subtype(row):
    er  = str(row.get('ER',  'na')).lower().strip()
    pr  = str(row.get('PR',  'na')).lower().strip()
    her = str(row.get('HER2','na')).lower().strip()
    ep = er in ['positive', '+']
    pp = pr in ['positive', '+']
    hp = her in ['positive', '+', 'amplified', 'equivocal']
    if not ep and not pp and not hp: return 'Triple_Negative'
    elif hp and (ep or pp): return 'Luminal_HER2+'
    elif hp: return 'HER2_Enriched'
    elif ep or pp: return 'Luminal_HR+'
    return 'Unknown'

df_clin_ml['ER_PR_HER2_Combo'] = df_clin_ml.apply(classify_subtype, axis=1)

# Feature engineering
if 'age_at_diagnosis.diagnoses' in df_clin_ml.columns:
    if (df_clin_ml['age_at_diagnosis.diagnoses'].dropna() > 1000).any():
        df_clin_ml['age_at_diagnosis.diagnoses'] /= 365.25

avail_descriptors = [c for c in CLINICAL_DESCRIPTORS if c in df_clin_ml.columns]
print(f'Available clinical descriptors: {len(avail_descriptors)}')

df_clin_sub = df_clin_ml[['Model'] + avail_descriptors].copy()
# Encode label columns
for col in ['ER_PR_HER2_Combo', 'Subtype']:
    if col in df_clin_sub.columns:
        le = LabelEncoder()
        df_clin_sub[f'{col}_enc'] = le.fit_transform(df_clin_sub[col].fillna('Unknown').astype(str))

num_cols = df_clin_sub.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in df_clin_sub.select_dtypes(include=['object','category']).columns
            if c != 'Model']

for col in num_cols:
    df_clin_sub[col] = SimpleImputer(strategy='constant', fill_value=0).fit_transform(
        df_clin_sub[[col]])
for col in cat_cols:
    df_clin_sub[col] = df_clin_sub[col].fillna('Missing').astype(str)

import sklearn
ohe_kw = {'handle_unknown':'ignore', 'sparse_output':False} if tuple(int(x) for x in sklearn.__version__.split('.')[:2]) >= (1,2) else {'handle_unknown':'ignore','sparse':False}

transformers = []
if num_cols: transformers.append(('num', StandardScaler(), num_cols))
if cat_cols: transformers.append(('cat', OneHotEncoder(**ohe_kw), cat_cols))

if transformers:
    ct = ColumnTransformer(transformers, remainder='drop')
    X_clin = ct.fit_transform(df_clin_sub)
    print(f'Clinical feature matrix: {X_clin.shape}')
else:
    X_clin = np.zeros((len(df_clin_sub), 1))
    print('⚠️  No clinical features available, using placeholder')

ids_clin = df_clin_sub['Model'].values

In [ ]:
# ── 6.2 UMAP + clustering on clinical data ───────────────────────────────
CLIN_N_NEIGHBORS = [10, 30, 50]
CLIN_MIN_DIST    = [0.05, 0.3]
CLIN_N_COMPONENTS= [2, 3]
CLIN_METRICS     = ['euclidean', 'cosine']

emb_clin = {}
for seed in SEEDS:
    np.random.seed(seed)
    for nc in CLIN_N_COMPONENTS:
        for nn in CLIN_N_NEIGHBORS:
            for md in CLIN_MIN_DIST:
                for mt in CLIN_METRICS:
                    key = f'UMAP_C{nc}_NN{nn}_MD{md}_M{mt}_S{seed}'
                    try:
                        r = umap.UMAP(n_components=nc, n_neighbors=nn,
                                      min_dist=md, metric=mt,
                                      random_state=seed, n_jobs=1, verbose=False)
                        emb_clin[key] = r.fit_transform(X_clin)
                    except:
                        pass

print(f'Clinical UMAP embeddings: {len(emb_clin)}')

bw_c = estimate_bandwidth(X_clin, quantile=0.2, n_samples=min(500, len(X_clin))) or 1.0

alg_configs_clin = {
    'KMeans':       (KMeans, ParameterGrid({'n_clusters': range(2,6)}), True),
    'Agglomerative':(AgglomerativeClustering, ParameterGrid({'n_clusters': range(2,6), 'linkage':['ward','average']}), False),
    'GMM':          (GaussianMixture, ParameterGrid({'n_components': range(2,5)}), True),
    'DBSCAN':       (DBSCAN, ParameterGrid({'eps':[0.5,1.0,1.5], 'min_samples':[5,10]}), False),
    'MeanShift':    (MeanShift, ParameterGrid({'bandwidth':[bw_c, bw_c*1.5, bw_c*0.5]}), False),
    'BayesGMM':     (BayesianGaussianMixture, ParameterGrid({'n_components': range(2,5)}), True),
}

print('Running clinical clustering sweep...')
df_clin_clust = run_clustering_sweep(emb_clin, alg_configs_clin, SEEDS, MAX_NOISE_PCT)
df_clin_selected = (df_clin_clust[df_clin_clust['score'] >= SILHOUETTE_THRESHOLD]
                    .sort_values(['score','chi','db'], ascending=[False,False,True])
                    .reset_index(drop=True))
print(f'Valid clinical clusterings: {len(df_clin_selected)}')
if not df_clin_selected.empty:
    top_c = df_clin_selected.iloc[0]
    print(f'Best: {top_c.algorithm} | {top_c.reduction} | Sil={top_c.score:.4f} | K={top_c.nk}')

# Export
df_clusters_clin = pd.DataFrame({'Model': ids_clin})
for i,row in df_clin_selected.iterrows():
    col = f"Cluster_{row.reduction}_{row.algorithm}_K{row.nk}_S{row.score:.2f}_Seed{row.seed}"
    if len(row.labels) == len(ids_clin):
        df_clusters_clin[col] = row.labels
clin_save = os.path.join(RESULTS_DIR, 'PatientClusters_Clinical.csv')
df_clusters_clin.to_csv(clin_save, index=False)
print(f'Saved: {clin_save}')

In [ ]:
# ── 6.3 Figure: Clinical cluster 3D visualization (Fig 9) ────────────────
if not df_clin_selected.empty:
    best_c = df_clin_selected.iloc[0]
    emb_key = best_c.reduction
    lbl     = best_c.labels.astype(str)
    X_ec    = emb_clin[emb_key]
    ndim    = X_ec.shape[1]

    unique_cl = sorted(set(lbl))
    valid_cl  = [c for c in unique_cl if c != '-1']
    cmap_c    = {c: CLUSTER_COLORS[i % len(CLUSTER_COLORS)] for i,c in enumerate(valid_cl)}
    if '-1' in unique_cl: cmap_c['-1'] = PALETTE['neutral']

    if ndim >= 3:
        fig = plt.figure(figsize=(W_HALF, 5.5))
        ax3 = fig.add_subplot(111, projection='3d')
        x_ax = np.arange(len(ids_clin))
        for cl in unique_cl:
            mask = lbl == cl
            ax3.scatter(x_ax[mask], X_ec[mask,0], X_ec[mask,1],
                        color=cmap_c[cl], label=f'Cluster {cl}', alpha=0.75, s=15)
        ax3.set_xlabel('Patient Index'); ax3.set_ylabel('UMAP 1'); ax3.set_zlabel('UMAP 2')
    else:
        fig, ax3 = plt.subplots(figsize=(W_HALF, 5.5))
        for cl in unique_cl:
            mask = lbl == cl
            ax3.scatter(X_ec[mask,0], X_ec[mask,1], color=cmap_c[cl],
                        label=f'Cluster {cl}', alpha=0.75, s=20, edgecolors='none')
        ax3.set_xlabel('UMAP 1'); ax3.set_ylabel('UMAP 2')

    plt.title(f'Clinical Patient Clustering — 3D View\n'
              f'Silhouette={best_c.score:.3f} | K={best_c.nk}',
              fontsize=FONT_TITLE, fontweight='bold')
    plt.legend(title='Cluster', fontsize=FONT_SIZE)
    plt.tight_layout()
    savefig('Fig09_clinical_cluster_3D.png')

## 7. Cross-Modal Concordance Analysis (ARI / AMI)

Compares metabolic vs. clinical cluster assignments.

- **Metric:** Adjusted Rand Index (ARI) — chance-corrected agreement
- All clinical × metabolic algorithm pairs are evaluated
- Top 10 pairs shown in heatmap (Fig 8 in paper)

> Partial alignment suggests metabolic states capture information that overlaps with,
> but is not fully explained by, clinical classifications.

In [ ]:
# ── 7.1 Build merged cluster matrix ─────────────────────────────────────
df_meta_cl = df_clusters_meta.rename(columns={'Model':'ModelName'})
df_clin_cl = df_clusters_clin.rename(columns={'Model':'ModelName'})

# Standardize IDs
df_meta_cl['ModelName'] = df_meta_cl['ModelName'].astype(str).str.split('_').str[0].str.slice(0,16)
df_clin_cl['ModelName'] = df_clin_cl['ModelName'].astype(str).str.split('_').str[0].str.slice(0,16)

# Rename columns to avoid ambiguity
meta_cluster_cols = [c for c in df_meta_cl.columns if c != 'ModelName']
clin_cluster_cols = [c for c in df_clin_cl.columns if c != 'ModelName']

df_meta_cl.columns = ['ModelName'] + [f'{c}_M' for c in meta_cluster_cols]
df_clin_cl.columns = ['ModelName'] + [f'{c}_C' for c in clin_cluster_cols]

df_merged_clusters = df_clin_cl.merge(df_meta_cl, on='ModelName', how='inner')
print(f'Common patients: {len(df_merged_clusters)}')

meta_cols_m = [c for c in df_merged_clusters.columns if c.endswith('_M')]
clin_cols_c = [c for c in df_merged_clusters.columns if c.endswith('_C')]
print(f'Metabolic cluster columns: {len(meta_cols_m)}')
print(f'Clinical cluster columns:  {len(clin_cols_c)}')

In [ ]:
# ── 7.2 Compute ARI/AMI for all pairs ───────────────────────────────────
MIN_VALID = 50
results_ari = []

for c_col in clin_cols_c:
    for m_col in meta_cols_m:
        s1 = df_merged_clusters[c_col].replace(-1, np.nan)
        s2 = df_merged_clusters[m_col].replace(-1, np.nan)
        valid_idx = s1.dropna().index.intersection(s2.dropna().index)
        if len(valid_idx) < MIN_VALID: continue
        l1 = s1.loc[valid_idx].astype(int)
        l2 = s2.loc[valid_idx].astype(int)
        if l1.nunique() < 2 or l2.nunique() < 2: continue
        ari = adjusted_rand_score(l1, l2)
        ami = adjusted_mutual_info_score(l1, l2)
        results_ari.append({'Clinical_Cluster': c_col, 'Metabolic_Cluster': m_col,
                             'ARI': ari, 'AMI': ami, 'N': len(valid_idx)})

df_ari = pd.DataFrame(results_ari).sort_values('ARI', ascending=False)
print(f'ARI pairs computed: {len(df_ari)}')
print(f'\nTop 5 concordances:')
print(df_ari.head(5)[['ARI','AMI','N']].to_string())

In [ ]:
# ── 7.3 Figure: Top-10 concordance heatmap (Fig 8 in paper) ─────────────
# Short name extractors
def short_name(col, suffix):
    # Extract algorithm name and key metrics
    m = re.search(r'_(KMeans|Agglomerative|DBSCAN|GMM|BayesGMM|MeanShift|HDBSCAN)_', col)
    algo = m.group(1) if m else 'Unknown'
    m2 = re.search(r'_K(\d+)_', col)
    k = m2.group(1) if m2 else '?'
    m3 = re.search(r'_S([0-9.]+)_', col)
    sil = m3.group(1) if m3 else ''
    return f'{algo} K{k} Sil{sil} [{suffix}]'

df_top10 = df_ari.head(10).copy()
df_top10['Clinical_Short']  = df_top10['Clinical_Cluster'].apply(lambda c: short_name(c, 'C'))
df_top10['Metabolic_Short'] = df_top10['Metabolic_Cluster'].apply(lambda c: short_name(c, 'M'))

# Best metabolic algorithm label
best_meta_lbl = df_top10['Metabolic_Short'].iloc[0]
df_top10 = df_top10.sort_values('ARI', ascending=True)

fig, ax = plt.subplots(figsize=(W_FULL * 0.85, 6.5))
sns.heatmap(
    df_top10[['ARI']].values,
    annot=df_top10['ARI'].values.reshape(-1,1),
    fmt='.3f',
    cmap='magma',
    vmin=0, vmax=1,
    yticklabels=df_top10['Clinical_Short'],
    xticklabels=[best_meta_lbl],
    linewidths=0.6, linecolor='white',
    annot_kws={'size': FONT_SIZE, 'weight': 'bold'},
    cbar_kws={'label': 'Adjusted Rand Index', 'shrink': 0.6},
    ax=ax)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=FONT_SIZE - 1, rotation=0)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=FONT_SIZE - 1, rotation=15, ha='right')
ax.set_title('Top 10 Concordances between Clinical and Metabolic Clustering',
             fontsize=FONT_TITLE, fontweight='bold', pad=10)
ax.set_ylabel('Clinical Algorithm', fontsize=FONT_AXIS, fontweight='bold')
ax.set_xlabel('Metabolic Algorithm', fontsize=FONT_AXIS, fontweight='bold')
plt.tight_layout()
savefig('Fig08_top10_ARI_heatmap.png')

## 8. Core / Divergent Patient Group Identification

Identify the Divergent subgroup (~3.3% of patients with globally reduced metabolic activity).

In [ ]:
# ── 8.1 Find best clinical-metabolic pair ────────────────────────────────
if not df_ari.empty:
    best_pair   = df_ari.iloc[0]
    best_c_col  = best_pair['Clinical_Cluster']
    best_m_col  = best_pair['Metabolic_Cluster']
    print(f'Best pair: ARI={best_pair.ARI:.4f}')
    print(f'  Clinical:  {best_c_col}')
    print(f'  Metabolic: {best_m_col}')

    df_best = df_merged_clusters[['ModelName', best_c_col, best_m_col]].copy()
    df_best = df_best[(df_best[best_c_col] != -1) & (df_best[best_m_col] != -1)]

    # Contingency matrix
    contingency = pd.crosstab(df_best[best_c_col], df_best[best_m_col])
    stacked = contingency.stack()
    c_best, m_best = stacked.idxmax()
    n_core = stacked.max()
    n_total = len(df_best)

    print(f'\nCore group: Clinical={c_best}, Metabolic={m_best}')
    print(f'Core size: {n_core} / {n_total} ({n_core/n_total*100:.1f}%)')

    # Classify patients
    def categorize(row):
        if   row[best_c_col] == c_best and row[best_m_col] == m_best: return 'Core (Concordant)'
        elif row[best_c_col] == c_best and row[best_m_col] != m_best: return 'Metabolic Divergence'
        elif row[best_c_col] != c_best and row[best_m_col] == m_best: return 'Clinical Divergence'
        else: return 'Total Discrepancy'

    df_best['Category'] = df_best.apply(categorize, axis=1)
    print('\nCohort composition:')
    print(df_best['Category'].value_counts())

    # Divergent = those NOT in the Core
    df_core      = df_best[df_best['Category'] == 'Core (Concordant)']
    df_divergent = df_best[df_best['Category'] != 'Core (Concordant)']

    df_core.to_csv(os.path.join(RESULTS_DIR, 'core_patients.csv'), index=False)
    df_divergent.to_csv(os.path.join(RESULTS_DIR, 'divergent_patients.csv'), index=False)
    print(f'\n✅ Core: {len(df_core)} | Divergent: {len(df_divergent)}')

else:
    print('⚠️  No ARI results. Using metabolic cluster directly.')
    best_meta_col = [c for c in df_clusters_meta.columns if c != 'Model'][0]
    lbl_arr = df_clusters_meta[best_meta_col].values
    counts = pd.Series(lbl_arr).value_counts()
    major_cl = counts.index[0]
    core_ids = df_clusters_meta.loc[df_clusters_meta[best_meta_col] == major_cl, 'Model']
    div_ids  = df_clusters_meta.loc[df_clusters_meta[best_meta_col] != major_cl, 'Model']
    df_core      = pd.DataFrame({'ModelName': core_ids})
    df_divergent = pd.DataFrame({'ModelName': div_ids})
    print(f'Core: {len(df_core)} | Divergent: {len(df_divergent)}')

In [ ]:
# ── 8.2 Figure: Contingency heatmap (Fig 2 equivalent) ───────────────────
if not df_ari.empty and 'contingency' in dir():
    fig, ax = plt.subplots(figsize=(W_HALF * 0.8, 4.5))
    sns.heatmap(contingency, annot=True, fmt='d', cmap='YlGnBu',
                linewidths=0.5, linecolor='white',
                annot_kws={'size': FONT_SIZE, 'weight': 'bold'},
                cbar_kws={'label': 'Number of Patients', 'shrink': 0.8}, ax=ax)
    ax.set_title('Patient Distribution: Clinical vs Metabolic Clusters',
                 fontsize=FONT_TITLE, fontweight='bold', pad=10)
    ax.set_xlabel('Metabolic Clusters', fontsize=FONT_AXIS, fontweight='bold')
    ax.set_ylabel('Clinical Clusters',  fontsize=FONT_AXIS, fontweight='bold')
    plt.tight_layout()
    savefig('Fig02_contingency_matrix.png')

In [ ]:
# ── 8.3 Figure: Cohort pie chart (Fig 3) ─────────────────────────────────
if not df_ari.empty:
    group_summary = df_best['Category'].value_counts()
    colors_pie = [PALETTE['core'], PALETTE['divergent'], PALETTE['accent1'], PALETTE['neutral']]
    n_cat = len(group_summary)
    explode = ([0.05, 0.15, 0.10, 0.08] + [0.05]*n_cat)[:n_cat]

    fig, ax = plt.subplots(figsize=(W_HALF, 5))
    wedges, texts, autotexts = ax.pie(
        group_summary.values, autopct='%1.1f%%', startangle=140,
        colors=colors_pie[:n_cat], explode=explode,
        pctdistance=0.72, wedgeprops={'linewidth':0.8,'edgecolor':'white'})
    for at in autotexts:
        at.set_fontsize(FONT_SIZE); at.set_fontweight('bold')
    ax.legend(group_summary.index, loc='lower left', fontsize=FONT_SIZE,
              title='Category', title_fontsize=FONT_SIZE)
    ax.set_title('Cohort Composition: Concordance vs Divergence',
                 fontsize=FONT_TITLE, fontweight='bold', pad=12)
    plt.tight_layout()
    savefig('Fig03_cohort_pie.png')

## 9. Divergent Subgroup Characterization — Cliff's Delta (Fig 11 in paper)

In [ ]:
# ── 9.1 Assign Core/Divergent labels to feature matrix ──────────────────
# Build label vector for metabolic features
core_ids = set(df_core['ModelName'].astype(str).str.slice(0,16).tolist()) if 'df_core' in dir() else set()
div_ids  = set(df_divergent['ModelName'].astype(str).str.slice(0,16).tolist()) if 'df_divergent' in dir() else set()

# If we have no groups, fall back to metabolic cluster labels
if len(core_ids) == 0 and not df_clust_selected.empty:
    top_meta = df_clust_selected.iloc[0]
    lbl_meta = top_meta.labels
    cnt = pd.Series(lbl_meta).value_counts()
    major = cnt.index[0]
    core_ids = set(ids_tumor[lbl_meta == major])
    div_ids  = set(ids_tumor[lbl_meta != major])

# Index into full feature matrix (tumor only)
group_labels = np.array([
    'Core' if m in core_ids else ('Divergent' if m in div_ids else 'Unknown')
    for m in ids_tumor
])

mask_core = group_labels == 'Core'
mask_div  = group_labels == 'Divergent'
X_core = X_tumor[mask_core]
X_div  = X_tumor[mask_div]

print(f'Core samples:      {mask_core.sum()}')
print(f'Divergent samples: {mask_div.sum()}')

In [ ]:
# ── 9.2 Cliff's delta + Mann-Whitney (FDR) ───────────────────────────────
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

def cliffs_delta(x, y):
    '''Compute Cliff's delta effect size.'''
    n1, n2 = len(x), len(y)
    if n1 == 0 or n2 == 0: return 0.0
    d = sum(1 if xi > yj else (-1 if xi < yj else 0)
            for xi in x for yj in y)
    return d / (n1 * n2)

# Use vectorized approximation for speed
def cliffs_delta_fast(x, y):
    n1, n2 = len(x), len(y)
    if n1==0 or n2==0: return 0.0
    # U statistic → delta
    _, pval = mannwhitneyu(x, y, alternative='two-sided')
    u1 = sum(np.sum(xi > y) for xi in x)
    return (2*u1 - n1*n2) / (n1*n2)

delta_results = []
pvals_delta   = []

for j, feat_name in enumerate(metabolic_feat_names):
    xc = X_core[:, j]
    xd = X_div[:, j]
    if len(xc) < 2 or len(xd) < 2:
        continue
    delta = cliffs_delta_fast(xc, xd)
    _, pval = mannwhitneyu(xc, xd, alternative='two-sided')
    delta_results.append({'feature': feat_name, 'delta': delta, 'pval': pval})
    pvals_delta.append(pval)

df_delta = pd.DataFrame(delta_results)
if len(df_delta) > 0:
    _, padj, _, _ = multipletests(df_delta['pval'], method='fdr_bh', alpha=0.05)
    df_delta['pval_adj'] = padj
    df_delta['significant'] = padj < 0.05

    def effect_label(d):
        ad = abs(d)
        if ad >= 0.474: return '***'
        elif ad >= 0.33: return '**'
        elif ad >= 0.147: return '*'
        return ''

    df_delta['effect_label'] = df_delta['delta'].apply(effect_label)
    df_delta_sig = df_delta[df_delta['significant']].sort_values('delta')
    print(f'Significant features (FDR<0.05): {len(df_delta_sig)}')
    df_delta.to_csv(os.path.join(RESULTS_DIR, 'CliffsDelta_CoreVsDivergent.csv'), index=False)
else:
    print('⚠️  Insufficient samples for Cliff\'s delta')
    df_delta_sig = pd.DataFrame()

In [ ]:
# ── 9.3 Figure: Forest plot Cliff's delta (Fig 11 in paper) ──────────────
if len(df_delta_sig) >= 2:
    n_show = min(40, len(df_delta_sig))
    df_top40 = df_delta_sig.iloc[:n_show].copy()

    # Short feature names
    def short_feat(name):
        name = re.sub(r'_(FBA|pFBA|L1w|L2)$', r' (\1)', name)
        name = re.sub(r'^SA_', 'SA: ', name)
        name = re.sub(r'^Oncomet_', '', name)
        return name[:55]

    df_top40['short_name'] = df_top40['feature'].apply(short_feat)

    fig, ax = plt.subplots(figsize=(W_FULL * 0.75, n_show * 0.28 + 1.5))
    colors_forest = [PALETTE['divergent'] if d < 0 else PALETTE['core'] for d in df_top40['delta']]

    bars = ax.barh(range(n_show), df_top40['delta'].values,
                   color=colors_forest, alpha=0.8, height=0.7,
                   edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=1.0, linestyle='-')

    for i, (_, row) in enumerate(df_top40.iterrows()):
        x = row['delta']
        offset = 0.01 if x >= 0 else -0.01
        ha = 'left' if x >= 0 else 'right'
        ax.text(x + offset, i, f"{row['effect_label']} {abs(row['delta']):.2f}",
                va='center', ha=ha, fontsize=FONT_SIZE - 2)

    ax.set_yticks(range(n_show))
    ax.set_yticklabels(df_top40['short_name'], fontsize=FONT_SIZE - 1)
    ax.set_xlabel("Cliff's Delta (negative = higher in Core)",
                  fontsize=FONT_AXIS, fontweight='bold')
    ax.set_title(f"Top {n_show} Metabolic Features — Cliff's Delta\n"
                 f"Core (n={mask_core.sum()}) vs Divergent (n={mask_div.sum()})",
                 fontsize=FONT_TITLE, fontweight='bold')

    from matplotlib.patches import Patch
    legend_handles = [
        Patch(color=PALETTE['core'],      label='Core > Divergent'),
        Patch(color=PALETTE['divergent'], label='Divergent > Core'),
    ]
    ax.legend(handles=legend_handles, fontsize=FONT_SIZE, loc='lower right')
    ax.invert_yaxis()
    plt.tight_layout()
    savefig('Fig11_CliffsDelta_forest_plot.png')

## 10. Kaplan–Meier Survival Analysis (Fig 12 in paper)

In [ ]:
# ── 10.1 Survival analysis ───────────────────────────────────────────────
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# Merge survival data with group labels
df_surv_groups = pd.DataFrame({'Model': ids_tumor, 'Group': group_labels})
df_surv_groups['Model'] = df_surv_groups['Model'].str.slice(0, 16)

if not df_survival.empty:
    df_surv_m = df_survival.copy()
    df_surv_m['Model'] = df_surv_m['Model'].str.slice(0, 16)
    df_surv_km = df_surv_groups.merge(df_surv_m[['Model','OS.time','OS']],
                                       on='Model', how='inner')
    df_surv_km = df_surv_km[(df_surv_km['Group'].isin(['Core','Divergent'])) &
                              df_surv_km['OS.time'].notna() &
                              df_surv_km['OS'].notna()]
    df_surv_km['OS.time'] = df_surv_km['OS.time'].astype(float)
    df_surv_km['OS']      = df_surv_km['OS'].astype(int)

    df_core_km = df_surv_km[df_surv_km['Group'] == 'Core']
    df_div_km  = df_surv_km[df_surv_km['Group'] == 'Divergent']
    print(f'Core for KM: {len(df_core_km)}')
    print(f'Divergent for KM: {len(df_div_km)}')

    if len(df_core_km) >= 5 and len(df_div_km) >= 2:
        results_lr = logrank_test(
            df_core_km['OS.time'], df_div_km['OS.time'],
            event_observed_A=df_core_km['OS'],
            event_observed_B=df_div_km['OS'])
        p_val = results_lr.p_value
        print(f'Log-rank test p-value: {p_val:.4f}')

        # Plot
        fig, ax = plt.subplots(figsize=(W_HALF, 5))

        kmf_core = KaplanMeierFitter(label=f'Core (n={len(df_core_km)})')
        kmf_core.fit(df_core_km['OS.time'], event_observed=df_core_km['OS'])
        kmf_core.plot_survival_function(ax=ax, ci_show=True,
                                         color=PALETTE['core'], linewidth=2)

        kmf_div = KaplanMeierFitter(label=f'Divergent (n={len(df_div_km)})')
        kmf_div.fit(df_div_km['OS.time'], event_observed=df_div_km['OS'])
        kmf_div.plot_survival_function(ax=ax, ci_show=True,
                                        color=PALETTE['divergent'], linewidth=2)

        ax.set_xlabel('Time (days)', fontsize=FONT_AXIS, fontweight='bold')
        ax.set_ylabel('Survival Probability', fontsize=FONT_AXIS, fontweight='bold')
        ax.set_title(f'Kaplan–Meier Overall Survival Curves\n'
                     f'Log-rank p = {p_val:.3f}',
                     fontsize=FONT_TITLE, fontweight='bold')
        ax.legend(fontsize=FONT_SIZE)
        ax.set_ylim(0, 1.05)
        plt.tight_layout()
        savefig('Fig12_KaplanMeier_survival.png')
    else:
        print('⚠️  Insufficient samples for KM analysis')
else:
    print('⚠️  Survival data not available')

## 11. Metabolic Signatures Across Molecular Subtypes (Fig 13 in paper)

In [ ]:
# ── 11.1 Heatmap: metabolic features by subtype & cluster ───────────────
# Build combined dataframe: group + subtype + top features
df_heatmap_data = pd.DataFrame({'Model': ids_tumor, 'Group': group_labels})
df_heatmap_data['Model'] = df_heatmap_data['Model'].str.slice(0,16)

if not df_metadata.empty and 'Model' in df_metadata.columns:
    meta_sub = df_metadata[['Model','Subtype']].copy()
    meta_sub['Model'] = meta_sub['Model'].str.slice(0,16)
    df_heatmap_data = df_heatmap_data.merge(meta_sub, on='Model', how='left')
elif 'Subtype' in df_master.columns:
    sub_map = df_master.set_index('Model')['Subtype'].to_dict()
    df_heatmap_data['Subtype'] = df_heatmap_data['Model'].map(sub_map)
else:
    df_heatmap_data['Subtype'] = 'Unknown'

# Top features by Cliff's delta
if len(df_delta_sig) > 0:
    top_features = df_delta_sig.sort_values('delta').head(40)['feature'].tolist()
else:
    top_features = metabolic_feat_names[:20]

top_feat_idx = [metabolic_feat_names.index(f) for f in top_features if f in metabolic_feat_names]
top_feat_names = [metabolic_feat_names[i] for i in top_feat_idx]

# Group by subtype × cluster
valid_groups = ['Core', 'Divergent']
subtypes = df_heatmap_data['Subtype'].fillna('Unknown')
valid_subtypes = subtypes[df_heatmap_data['Group'].isin(valid_groups)].unique()

hm_rows = []
hm_labels = []
for group in valid_groups:
    for sub in sorted(valid_subtypes)[:8]:  # top 8 subtypes
        mask = (df_heatmap_data['Group'] == group) & (subtypes == sub)
        idx = np.where(mask)[0]
        if len(idx) < 2: continue
        row_median = np.median(X_tumor[idx][:, top_feat_idx], axis=0)
        hm_rows.append(row_median)
        hm_labels.append(f'{group}\n{sub}')

if len(hm_rows) >= 2:
    hm_matrix = np.array(hm_rows)
    # z-score
    hm_z = (hm_matrix - hm_matrix.mean(axis=0)) / (hm_matrix.std(axis=0) + 1e-9)

    short_feat_names = [re.sub(r'_(FBA|pFBA|L1w)$', r' (\1)', n)[:35] for n in top_feat_names]

    fig_h = max(6.0, len(hm_labels) * 0.45)
    fig, ax = plt.subplots(figsize=(W_FULL * 0.8, fig_h))
    sns.heatmap(hm_z, annot=False, cmap='RdBu_r', center=0, vmin=-2, vmax=2,
                xticklabels=short_feat_names, yticklabels=hm_labels,
                linewidths=0.3, linecolor='white',
                cbar_kws={'label': 'Z-score', 'shrink': 0.6}, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=FONT_SIZE - 2)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=FONT_SIZE - 1)
    ax.set_title('Metabolic Signatures Across Molecular Subtypes\n'
                 'Top Features by Effect Size (Cliff\'s Delta)',
                 fontsize=FONT_TITLE, fontweight='bold')
    ax.set_xlabel('Metabolic Feature', fontsize=FONT_AXIS, fontweight='bold')
    ax.set_ylabel('Group / Subtype', fontsize=FONT_AXIS, fontweight='bold')
    plt.tight_layout()
    savefig('Fig13_heatmap_metabolic_signatures_subtypes.png')
else:
    print('⚠️  Insufficient data for heatmap')

## 12. Pareto Surface Analysis (Multi-Objective Metabolic Optimization)

In [ ]:
# ── 12.1 Load Pareto data (if available via LFS) ────────────────────────
pareto_path_100  = os.path.join(DATA_DIR, 'ParetoSurface_100sol.csv')
pareto_path_1000 = os.path.join(DATA_DIR, 'ParetoSurface_1000sol.csv')

# Try to download from LFS
PARETO_RAW = f'{RAW}/Clinical_data_and_models_ids/Metabolic_Data/Archived'
for fname, url_name in [
    ('ParetoSurface_100sol.csv', 'ParetoSurface_CU_EA_extended_1226_Final_100soluciones.csv'),
    ('ParetoSurface_1000sol.csv', 'ParetoSurface_CU_EA_extended_1226_Final_ALL_1000sol.csv'),
]:
    dest = os.path.join(DATA_DIR, fname)
    if not os.path.exists(dest):
        try:
            urllib.request.urlretrieve(f'{PARETO_RAW}/{url_name}', dest)
            sz = os.path.getsize(dest)
            print(f'✅ {fname}: {sz/1e6:.1f} MB')
        except Exception as e:
            print(f'⚠️  {fname}: {e}')

df_pareto = None
for path in [pareto_path_100, pareto_path_1000]:
    if os.path.exists(path) and os.path.getsize(path) > 1000:
        df_pareto = pd.read_csv(path)
        print(f'Loaded Pareto data: {df_pareto.shape}')
        break

if df_pareto is None:
    print('ℹ️  Pareto data not available. Generating synthetic visualization from FeatureMatrix.')
    # Use CU and EA columns from feature matrix to simulate Pareto
    cu_cols = [c for c in df_feat.columns if c.startswith('CU_')]
    ea_cols = [c for c in df_feat.columns if c.startswith('EA_')]
    if cu_cols and ea_cols:
        df_pareto_sim = df_feat[['Model'] + cu_cols[:1] + ea_cols[:1]].copy()
        df_pareto_sim.columns = ['Model', 'CU', 'EA']
        df_pareto_sim['Model'] = df_pareto_sim['Model'].str.slice(0,16)
        df_pareto_sim['Group'] = df_pareto_sim['Model'].map(
            lambda m: 'Divergent' if m in div_ids else ('Core' if m in core_ids else 'Other'))
        print(f'Pareto simulation from feature matrix: {df_pareto_sim.shape}')
    else:
        df_pareto_sim = None
        print('⚠️  CU/EA columns not found in feature matrix')

In [ ]:
# ── 12.2 Figure: Pareto 2x2 panel (Fig in Supplementary) ────────────────
if df_pareto is not None:
    # Detect CU, EA, ATP columns in pareto data
    cu_col  = next((c for c in df_pareto.columns if 'CU' in c), None)
    ea_col  = next((c for c in df_pareto.columns if 'EA' in c), None)
    atp_col = next((c for c in df_pareto.columns if 'ATP' in c), None)
    bio_col = next((c for c in df_pareto.columns if 'Biomass' in c or 'biomass' in c), None)
    id_col  = next((c for c in df_pareto.columns if 'Model' in c or 'model' in c or
                    'Sample' in c), df_pareto.columns[0])

    df_pareto['_Group'] = df_pareto[id_col].astype(str).str.slice(0,16).map(
        lambda m: 'Divergent' if m in div_ids else ('Core' if m in core_ids else 'Other'))

    cmap_pg = {'Core': PALETTE['core'], 'Divergent': PALETTE['divergent'], 'Other': PALETTE['neutral']}
    cols_to_plot = [c for c in [cu_col, ea_col, atp_col, bio_col] if c is not None]

    if len(cols_to_plot) >= 2:
        n_panels = min(len(cols_to_plot), 4)
        fig, axes = plt.subplots(1, n_panels, figsize=(W_FULL, 4))
        if n_panels == 1: axes = [axes]

        # Sample 10 patients per group for Pareto display
        sampled_ids_core = df_pareto[df_pareto['_Group']=='Core'][id_col].unique()[:5]
        sampled_ids_div  = df_pareto[df_pareto['_Group']=='Divergent'][id_col].unique()[:5]
        sampled_ids = list(sampled_ids_core) + list(sampled_ids_div)

        for ax, col in zip(axes, cols_to_plot[:n_panels]):
            ref_col = cols_to_plot[0] if col != cols_to_plot[0] else (cols_to_plot[1] if len(cols_to_plot)>1 else col)
            for sid in sampled_ids:
                mask = df_pareto[id_col] == sid
                group = df_pareto.loc[mask, '_Group'].iloc[0] if mask.sum() > 0 else 'Other'
                color = cmap_pg.get(group, PALETTE['neutral'])
                alpha = 0.8 if group == 'Divergent' else 0.4
                ax.plot(df_pareto.loc[mask, ref_col], df_pareto.loc[mask, col],
                        color=color, alpha=alpha, linewidth=1.2, marker='o', markersize=2)
            ax.set_xlabel(ref_col, fontsize=FONT_SIZE, fontweight='bold')
            ax.set_ylabel(col, fontsize=FONT_SIZE, fontweight='bold')
            ax.set_title(f'{ref_col} vs {col}', fontsize=FONT_SIZE)

        from matplotlib.lines import Line2D
        legend_handles = [
            Line2D([0],[0], color=PALETTE['core'],      lw=2, label='Core'),
            Line2D([0],[0], color=PALETTE['divergent'], lw=2, label='Divergent'),
        ]
        fig.legend(handles=legend_handles, fontsize=FONT_SIZE, loc='upper right')
        fig.suptitle('Pareto Front Analysis: Multi-Objective Metabolic Trade-offs',
                     fontsize=FONT_TITLE, fontweight='bold')
        plt.tight_layout()
        savefig('Fig_Pareto_panel.png')

elif 'df_pareto_sim' in dir() and df_pareto_sim is not None:
    fig, ax = plt.subplots(figsize=(W_HALF, 4.5))
    for group, color in [('Core', PALETTE['core']), ('Divergent', PALETTE['divergent'])]:
        sub = df_pareto_sim[df_pareto_sim['Group'] == group]
        ax.scatter(sub['CU'], sub['EA'], c=color, label=group, alpha=0.6, s=15, edgecolors='none')
    ax.set_xlabel('Carbon Uptake (CU)', fontsize=FONT_AXIS, fontweight='bold')
    ax.set_ylabel('Enzymatic Activity (EA)', fontsize=FONT_AXIS, fontweight='bold')
    ax.set_title('CU vs EA by Metabolic Group', fontsize=FONT_TITLE, fontweight='bold')
    ax.legend(fontsize=FONT_SIZE)
    plt.tight_layout()
    savefig('Fig_CU_EA_scatter.png')

## 13. Metabolic Pathways — Differential Activity Summary (Fig 3 equivalent in paper)

In [ ]:
# ── 13.1 Figure: Top significant metabolic pathways (bubble chart) ───────
# Extract pathways (subsystem activities SA_*)
sa_feat_names = [f for f in metabolic_feat_names if f.startswith('SA_')]

if len(sa_feat_names) >= 5 and len(X_core) >= 2 and len(X_div) >= 2:
    pathway_results = []
    for feat in sa_feat_names:
        j = metabolic_feat_names.index(feat)
        xc, xd = X_core[:, j], X_div[:, j]
        if len(xc) < 2 or len(xd) < 2: continue
        _, pv = mannwhitneyu(xc, xd, alternative='two-sided')
        delta = cliffs_delta_fast(xc, xd)
        # Extract pathway name (remove sol suffix)
        pathway = re.sub(r'_(FBA|pFBA|L1w)$', '', feat).replace('SA_', '').replace('_', ' ')
        pathway_results.append({'feature': feat, 'pathway': pathway,
                                  'pval': pv, 'delta': delta})

    df_path = pd.DataFrame(pathway_results)
    if len(df_path) > 0:
        _, padj_p, _, _ = multipletests(df_path['pval'], method='fdr_bh', alpha=0.05)
        df_path['pval_adj'] = padj_p
        df_path['neg_log_p'] = -np.log10(df_path['pval_adj'].clip(1e-300))
        df_path_sig = df_path[df_path['pval_adj'] < 0.05].sort_values('neg_log_p', ascending=False)
        print(f'Significant SA pathways: {len(df_path_sig)}')

        n_bubbles = min(15, len(df_path_sig))
        if n_bubbles >= 3:
            df_bubble = df_path_sig.head(n_bubbles)

            # Count reactions per pathway (proxy = number of SA entries sharing base name)
            base_names = df_bubble['pathway'].tolist()

            fig, ax = plt.subplots(figsize=(W_FULL * 0.7, 5))
            scatter = ax.scatter(
                df_bubble['neg_log_p'],
                range(n_bubbles),
                s=df_bubble['neg_log_p'] * 60 + 100,
                c=df_bubble['delta'],
                cmap='RdBu_r', vmin=-0.5, vmax=0.5,
                alpha=0.8, edgecolors='white', linewidths=0.5
            )
            cbar = plt.colorbar(scatter, ax=ax, shrink=0.6)
            cbar.set_label("Cliff's Delta", fontsize=FONT_SIZE)
            ax.set_yticks(range(n_bubbles))
            ax.set_yticklabels([n[:40] for n in base_names], fontsize=FONT_SIZE - 1)
            ax.set_xlabel('-log₁₀(adj. p-value)', fontsize=FONT_AXIS, fontweight='bold')
            ax.set_title('Significantly Altered Metabolic Pathways\nCore vs Divergent (FDR<0.05)',
                         fontsize=FONT_TITLE, fontweight='bold')
            ax.axvline(-np.log10(0.05), color='red', linestyle='--', alpha=0.5, linewidth=1)
            ax.invert_yaxis()
            plt.tight_layout()
            savefig('Fig03_metabolic_pathways_bubble.png')
else:
    print('ℹ️  Skipping pathway bubble chart (insufficient SA features or group sizes)')

## 14. GEM Size and Similarity Analysis (Fig 2 in paper)

In [ ]:
# ── 14.1 Figure: Model size distribution ─────────────────────────────────
# Check for model size columns in feature matrix
size_cols = [c for c in df_feat.columns if any(k in c.lower() for k in ['nrxn','nmet','ngene','size','reaction','metabolite','gene'])]
print('Size-related columns:', size_cols[:10])

# Try to detect n_reactions, n_metabolites, n_genes columns
rxn_col = next((c for c in size_cols if 'rx' in c.lower() or 'reaction' in c.lower()), None)
met_col = next((c for c in size_cols if 'met' in c.lower() or 'metabol' in c.lower()), None)
gene_col= next((c for c in size_cols if 'gene' in c.lower()), None)

if rxn_col:
    fig, axes = plt.subplots(1, min(3, len([c for c in [rxn_col, met_col, gene_col] if c])),
                              figsize=(W_FULL * 0.7, 4))
    if not hasattr(axes, '__len__'): axes = [axes]
    for ax, col_name, title in zip(axes,
                                   [c for c in [rxn_col, met_col, gene_col] if c],
                                   ['Reactions per Model', 'Metabolites per Model', 'Genes per Model']):
        data = df_feat[col_name].dropna()
        ax.hist(data, bins=30, color=PALETTE['core'], alpha=0.8, edgecolor='white')
        ax.axvline(data.mean(), color=PALETTE['divergent'], linestyle='--', linewidth=1.5,
                   label=f'Mean={data.mean():.0f}')
        ax.set_xlabel(title, fontsize=FONT_AXIS)
        ax.set_ylabel('Count', fontsize=FONT_AXIS)
        ax.set_title(title, fontsize=FONT_TITLE, fontweight='bold')
        ax.legend(fontsize=FONT_SIZE)
    plt.suptitle('Distribution of Patient-Specific GEM Sizes (n=1,226)',
                 fontsize=FONT_TITLE, fontweight='bold')
    plt.tight_layout()
    savefig('Fig02_model_size_distribution.png')
else:
    print('ℹ️  Model size columns not found in feature matrix — skipping Fig 2')

## 15. Final Summary — All Figures Generated

In [ ]:
# ── 15.1 List all output files ───────────────────────────────────────────
import glob

figures = sorted(glob.glob(os.path.join(RESULTS_DIR, 'Fig*.png')))
csvs    = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))

print('=== FIGURES GENERATED ===')
for f in figures:
    sz = os.path.getsize(f) / 1024
    print(f'  {os.path.basename(f):50s} {sz:6.1f} KB')

print(f'\n=== CSV FILES ===')
for f in csvs:
    sz = os.path.getsize(f) / 1024
    print(f'  {os.path.basename(f):50s} {sz:6.1f} KB')

print(f'\n=== SUMMARY ===')
print(f'Total tumor samples analyzed:  {mask_core.sum() + mask_div.sum() + (group_labels=="Unknown").sum()}')
print(f'Core patients:                 {mask_core.sum()}')
print(f'Divergent patients:            {mask_div.sum()}')
print(f'Divergent %:                   {mask_div.sum()/(mask_core.sum()+mask_div.sum())*100:.1f}% (paper: ~3.3%)')
if not df_clust_selected.empty:
    print(f'Best metabolic clustering Sil:  {df_clust_selected.iloc[0].score:.4f} (paper: ~0.98)')
if not df_clin_selected.empty:
    print(f'Best clinical clustering Sil:   {df_clin_selected.iloc[0].score:.4f} (paper: ~0.87)')
if not df_clf_results.empty:
    best_clf = df_clf_results.iloc[0]
    print(f'Best classifier:               {best_clf.Model} Acc={best_clf.Accuracy:.3f} (paper KNN: 0.988)')

In [ ]:
# ── 15.2 ZIP all results for download ────────────────────────────────────
import zipfile
zip_path = '/content/BRCA_metabolic_flux_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(os.path.join(RESULTS_DIR, '*')):
        zf.write(f, os.path.basename(f))
print(f'✅ Results zipped: {zip_path}')
print(f'   Size: {os.path.getsize(zip_path)/1024:.1f} KB')

# Download link in Colab
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('   (Download via Colab Files panel or: from google.colab import files; files.download("..."))')